In [44]:
import os
from dotenv import load_dotenv
from google import genai
load_dotenv(override=True)

True

In [46]:
"""
C221095_MD Minhaz Uddin
BudgiBird AI Agent
Agents are 
1. Router Agent
2. Knowledge Agent
3. Customer Support Agent
4. Bird Sales Agent
5. Refund Agent
6. Company Policy Agent
7. Emergency Agent
8. Image Analysis Agent
9. Final Response Agent
"""

# Imports

import os
import yaml
import base64
import mimetypes
from pathlib import Path
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from google import genai
from langsmith import wrappers
from langsmith import traceable

load_dotenv()

# Gemini Client
MODEL = "gemini-3.1-flash-lite"

gemini_client = genai.Client()

client = wrappers.wrap_gemini(
    gemini_client,
    tracing_extra={
        "project_name": "BudgiBird AI",
        "tags": [
            "BudgiBird",
            "Session4",
            "Multi-Agent",
            "Gemini"
        ],
        "metadata": {
            "author": "Minhaz",
            "version": "4.0"
        }
    }
)

# Load prompt.yml
PROMPTS = yaml.safe_load(
    Path("prompt.yml").read_text(encoding="utf-8")
)

ROUTER_SYSTEM = PROMPTS["router_system"]
KNOWLEDGE_SYSTEM = PROMPTS["knowledge_system"]
CUSTOMER_SYSTEM = PROMPTS["customer_support_system"]
SALES_SYSTEM = PROMPTS["sales_system"]
REFUND_SYSTEM = PROMPTS["refund_system"]
POLICY_SYSTEM = PROMPTS["policy_system"]
EMERGENCY_SYSTEM = PROMPTS["emergency_system"]
IMAGE_SYSTEM = PROMPTS["image_system"]
REPLY_SYSTEM = PROMPTS["reply_system"]

# Conversation Memory
history = []

INPUT_PRICE = 0.25
OUTPUT_PRICE = 1.50
token_usage = {
    "input": 0,
    "output": 0,
}

def track(usage):

    if usage is None:
        return
    token_usage["input"] += usage.prompt_token_count or 0
    token_usage["output"] += usage.candidates_token_count or 0


def usage_report():
    cost = (
        token_usage["input"] / 1_000_000 * INPUT_PRICE
        + token_usage["output"] / 1_000_000 * OUTPUT_PRICE
    )

    return (
        f"{token_usage['input']} input | "
        f"{token_usage['output']} output | "
        f"Estimated Cost = ${cost:.6f}"
    )

# Image(Base64)

def image_to_part(image_path: str):
    mime_type, _ = mimetypes.guess_type(image_path)
   
    if mime_type is None:
        mime_type = "image/jpeg"
    with open(image_path, "rb") as f:
        image = base64.b64encode(
            f.read()
        ).decode()

    return {
        "inline_data": {
            "mime_type": mime_type,
            "data": image
        }
    }

print("True")

True


In [47]:
# Router Output Schema
class Triage(BaseModel):
    """
    Structured output returned by the Router Agent.
    """

    route: Literal["knowledge","customer_support","bird_sales","refund","company_policy","emergency","image_analysis","general"] = Field(
        description="Which specialized agent should handle the request."
    )

    category: Literal[
        "diet","behavior","housing","health","disease", "breeding","mutation","sales","order","refund","policy","emergency","image", "general"
    ] = Field(
        description="Main topic of the user's request."
    )

    urgency: Literal["low","medium","high"
    ] = Field(
        description="Urgency level of the request."
    )

    needs_human: bool = Field(
        description="True if the user should be referred to a human staff member or avian veterinarian."
    )

    summary: str = Field(
        description="One short sentence summarizing the user's request."
    )


print("True")

True


In [48]:
# Dummy Customer Database
CUSTOMERS = {

    "C001": {
        "name": "Minhaz",
        "email": "minhaz@gmail.com",
        "phone": "017XXXXXXXX",
        "address": "Chattogram",

        "birds": [
            "B001",
            "B002"
        ],

        "orders": [
            "O001",
            "O002"
        ]
    },

    "C002": {
        "name": "Rahim",
        "email": "rahim@gmail.com",
        "phone": "018XXXXXXXX",
        "address": "Dhaka",

        "birds": [
            "B003"
        ],

        "orders": [
            "O003"
        ]
    }

}
# Bird Inventory
BIRDS = {

    "B001": {
        "mutation": "Albino",
        "gender": "Male",
        "age": "6 Months",
        "price": 3500,
        "status": "Sold"
    },

    "B002": {
        "mutation": "Lutino",
        "gender": "Female",
        "age": "5 Months",
        "price": 3200,
        "status": "Sold"
    },

    "B003": {
        "mutation": "Yellow Face",
        "gender": "Male",
        "age": "4 Months",
        "price": 2800,
        "status": "Sold"
    },

    "B004": {
        "mutation": "Rainbow",
        "gender": "Female",
        "age": "7 Months",
        "price": 4500,
        "status": "Available"
    },

    "B005": {
        "mutation": "Spangle",
        "gender": "Male",
        "age": "5 Months",
        "price": 3000,
        "status": "Available"
    }

}

# Orders
ORDERS = {

    "O001": {
        "customer_id": "C001",
        "bird_id": "B001",
        "status": "Delivered"
    },

    "O002": {
        "customer_id": "C001",
        "bird_id": "B002",
        "status": "Preparing"
    },

    "O003": {
        "customer_id": "C002",
        "bird_id": "B003",
        "status": "Delivered"
    }

}


# Refund Database
REFUNDS = {

    "R001": {
        "customer_id": "C001",
        "order_id": "O001",
        "status": "Pending"
    },

    "R002": {
        "customer_id": "C002",
        "order_id": "O003",
        "status": "Approved"
    }

}

# Company Policy

POLICIES = {

    "refund":
        "Refund is available within 3 days with a valid avian veterinarian certificate.",

    "replacement":
        "Replacement is available within 7 days for transport-related injuries.",

    "shipping":
        "Orders are usually delivered within 3 business days.",

    "warranty":
        "Bird health warranty is valid for 3 days after delivery."
}

print("True")

True


In [49]:
# Tool Layer
@traceable(name="Lookup Customer Tool")
def lookup_customer(customer_id: str):

    return CUSTOMERS.get(
        customer_id,
        {"error": "Customer not found."}
    )

@traceable(name="Lookup Order Tool")
def lookup_orders(customer_id: str):

    orders = []

    for order_id, order in ORDERS.items():

        if order["customer_id"] == customer_id:

            orders.append(
                {
                    "order_id": order_id,
                    **order
                }
            )

    return orders

@traceable(name="Lookup Refund Tool")
def lookup_refund(customer_id: str):

    refunds = []

    for refund_id, refund in REFUNDS.items():

        if refund["customer_id"] == customer_id:

            refunds.append(
                {
                    "refund_id": refund_id,
                    **refund
                }
            )

    return refunds

@traceable(name="Available Birds Tool")
def available_birds():

    birds = []

    for bird_id, bird in BIRDS.items():

        if bird["status"] == "Available":

            birds.append(
                {
                    "bird_id": bird_id,
                    **bird
                }
            )

    return birds

@traceable(name="Bird Price Tool")
def bird_price(bird_id: str):

    bird = BIRDS.get(bird_id)

    if bird is None:

        return {"error": "Bird not found."}

    return {

        "bird_id": bird_id,

        "mutation": bird["mutation"],

        "price": bird["price"]

    }


@traceable(name="Refund Policy Tool")
def refund_policy():

    return POLICIES["refund"]

@traceable(name="Replacement Policy Tool")
def replacement_policy():

    return POLICIES["replacement"]

@traceable(name="Shipping Policy Tool")
def shipping_policy():

    return POLICIES["shipping"]

@traceable(name="Warranty Policy Tool")
def warranty_policy():

    return POLICIES["warranty"]

@traceable(name="Knowledge Tool")
def knowledge_tool(category: str):

    knowledge = {

        "diet":
        "Budgies require pellets or seeds, leafy vegetables, fresh water, and fruits in moderation.",

        "behavior":
        "Budgies are social birds that need daily interaction and mental stimulation.",

        "housing":
        "Provide a spacious cage with natural perches and toys.",

        "breeding":
        "Breed only healthy adult budgies older than one year.",

        "mutation":
        "Popular mutations include Albino, Lutino, Rainbow, Spangle, and Yellow Face."

    }

    return knowledge.get(

        category,

        "General budgerigar care information."

    )

@traceable(name="Disease Tool")
def disease_tool(symptom: str):

    database = {

        "difficulty breathing":
        "Emergency. Visit an avian veterinarian immediately.",

        "bleeding":
        "Emergency. Apply gentle pressure and seek veterinary care.",

        "watery droppings":
        "Possible digestive illness or infection.",

        "sneezing":
        "Possible respiratory irritation or infection.",

        "feather loss":
        "May be molting, parasites, or nutritional deficiency."

    }

    return database.get(

        symptom.lower(),

        "No disease information available."

    )

@traceable(name="Emergency Tool")
def emergency_tool():

    return {

        "status": "Emergency",

        "action":
        "Visit an avian veterinarian immediately."

    }

@traceable(name="Company Policy Tool")
def company_policy():

    return POLICIES

@traceable(name="Image Tool")
def image_tool():

    return "Image received successfully and ready for Gemini Vision analysis."


print("True")

True


In [51]:
# Router Agent
@traceable(name="Router Agent")
def router_agent(message: str) -> Triage:
    """
    Analyze the user's message and determine which
    specialized agent should handle it.
    """

    response = client.models.generate_content(

        model=MODEL,

        contents=message,

        config={

            "system_instruction": ROUTER_SYSTEM,

            "response_mime_type": "application/json",

            "response_schema": Triage,

        }

    )

    track(response.usage_metadata)

    return response.parsed
print("True")

True


In [52]:
# Knowledge Agent
@traceable(name="Knowledge Agent")
def knowledge_agent(message: str, t: Triage):

    knowledge = knowledge_tool(t.category)

    return {
        "agent": "Knowledge Agent",
        "knowledge": knowledge,
    }

# Customer Support Agent

@traceable(name="Customer Support Agent")
def customer_support_agent(customer_id: str = None):

    # Guest User
    if customer_id is None:

        return {

            "agent": "Customer Support Agent",

            "status": "Need Customer ID",

            "message": "Please provide your Customer ID to access your account."

        }

    customer = lookup_customer(customer_id)

    if "error" in customer:

        return {

            "agent": "Customer Support Agent",

            "status": "Customer Not Found",

            "message": "Invalid Customer ID."

        }

    orders = lookup_orders(customer_id)

    return {

        "agent": "Customer Support Agent",

        "customer": customer,

        "orders": orders,

    }

# Bird Sales Agent
@traceable(name="Bird Sales Agent")
def bird_sales_agent():

    birds = available_birds()

    return {

        "agent": "Bird Sales Agent",

        "available_birds": birds

    }


# Refund Agent

@traceable(name="Refund Agent")
def refund_agent(customer_id: str = None):

    # Guest User
    if customer_id is None:

        return {

            "agent": "Refund Agent",

            "status": "Need Customer ID",

            "message": "Please provide your Customer ID to check your refund request."

        }

    customer = lookup_customer(customer_id)

    if "error" in customer:

        return {

            "agent": "Refund Agent",

            "status": "Customer Not Found",

            "message": "Invalid Customer ID."

        }

    refunds = lookup_refund(customer_id)

    policy = refund_policy()

    return {

        "agent": "Refund Agent",

        "refunds": refunds,

        "policy": policy

    }

# Company Policy Agent

@traceable(name="Company Policy Agent")
def company_policy_agent():

    policies = company_policy()

    return {

        "agent": "Company Policy Agent",

        "policies": policies

    }

# Emergency Agent
@traceable(name="Emergency Agent")
def emergency_agent(message: str):

    emergency = emergency_tool()

    disease = disease_tool(message)

    return {

        "agent": "Emergency Agent",

        "emergency": emergency,

        "possible_cause": disease

    }

# Image Analysis Agent
@traceable(name="Image Analysis Agent")
def image_analysis_agent():

    result = image_tool()

    return {

        "agent": "Image Analysis Agent",

        "result": result

    }


print("true")

true


In [53]:
# Final Response Agent
@traceable(name="Final Response Agent")
def final_response_agent(
    message: str,
    tool_result: dict,
    image_path: str = None
):

    prompt = f"""
User Question:
{message}

Tool Output:
{tool_result}

Instructions:
- Use ONLY the tool output.
- Never invent customer information.
- Never invent bird information.
- Never invent refund decisions.
- Be friendly and concise.
"""

    parts = [
        {
            "text": prompt
        }
    ]

    # Optional image
    if image_path is not None:
        parts.append(
            image_to_part(image_path)
        )

    response = client.models.generate_content(

        model=MODEL,

        contents=parts,

        config={
            "system_instruction": REPLY_SYSTEM
        }

    )

    track(response.usage_metadata)

    return response.text


print("true")

true


In [54]:
# Supervisor Agent
@traceable(name="Supervisor Agent")
def run_agent(
    message: str,
    customer_id: str = None,
    image_path: str = None
):

    print("CUSTOMER :", customer_id if customer_id else "Guest User")

    print("QUESTION :", message)

    # Step 1 : Router
    triage = router_agent(message)
    print("\nROUTER")
    print(triage.model_dump())

    # Step 2 : Select Agen
    if triage.route == "knowledge":
        tool_result = knowledge_agent(
            message,
            triage
        )

    elif triage.route == "customer_support":
        tool_result = customer_support_agent(
            customer_id
        )

    elif triage.route == "bird_sales":
        tool_result = bird_sales_agent()

    elif triage.route == "refund":
        tool_result = refund_agent(
            customer_id
        )

    elif triage.route == "company_policy":
        tool_result = company_policy_agent()

    elif triage.route == "emergency":
        tool_result = emergency_agent(
            message
        )

    elif triage.route == "image_analysis":
        tool_result = image_analysis_agent()

    else:

        tool_result = {
            "agent": "General",
            "message": "General conversation."
        }

    # Step 3 : Save User Message

    history.append({
        "role": "user",
        "customer_id": customer_id if customer_id else "Guest",
        "message": message
    })


    # Step 4 : Gemini Final Agent


    answer = final_response_agent(
        message=message,
        tool_result=tool_result,
        image_path=image_path
    )
    # Step 5 : Save AI Reply
    history.append({
        "role": "assistant",
        "message": answer

    })

    # Step 6 : Display Result
    print("\nSELECTED AGENT")
    print(tool_result["agent"])
    print("\nTOOL RESULT")
    print(tool_result)
    print("\nBUDGIBIRD AI")
    print(answer)
    print("\n")
    print("Conversation Length :", len(history))
    print("Token Usage :", usage_report())

In [55]:
run_agent(
    message="Can my budgie eat cucumber?"
)

CUSTOMER : Guest User
QUESTION : Can my budgie eat cucumber?

ROUTER
{'route': 'knowledge', 'category': 'diet', 'urgency': 'low', 'needs_human': False, 'summary': 'User is asking if cucumber is safe for budgies to consume.'}

SELECTED AGENT
Knowledge Agent

TOOL RESULT
{'agent': 'Knowledge Agent', 'knowledge': 'Budgies require pellets or seeds, leafy vegetables, fresh water, and fruits in moderation.'}

BUDGIBIRD AI
Yes, budgies can enjoy cucumber as part of their fresh vegetable intake in moderation!


Conversation Length : 2
Token Usage : 502 input | 73 output | Estimated Cost = $0.000235


In [56]:
run_agent(
    message="my birds is bleeding"
)

CUSTOMER : Guest User
QUESTION : my birds is bleeding

ROUTER
{'route': 'emergency', 'category': 'emergency', 'urgency': 'high', 'needs_human': True, 'summary': 'User reports their bird is bleeding and requires immediate assistance.'}

SELECTED AGENT
Emergency Agent

TOOL RESULT
{'agent': 'Emergency Agent', 'emergency': {'status': 'Emergency', 'action': 'Visit an avian veterinarian immediately.'}, 'possible_cause': 'No disease information available.'}

BUDGIBIRD AI
This sounds like an emergency; please take your bird to an avian veterinarian immediately.


Conversation Length : 4
Token Usage : 1007 input | 143 output | Estimated Cost = $0.000466


In [57]:
run_agent(
    message="i want to buy birds. what are available?"
)

CUSTOMER : Guest User
QUESTION : i want to buy birds. what are available?

ROUTER
{'route': 'bird_sales', 'category': 'sales', 'urgency': 'low', 'needs_human': False, 'summary': 'The user is inquiring about available birds for purchase.'}

SELECTED AGENT
Bird Sales Agent

TOOL RESULT
{'agent': 'Bird Sales Agent', 'available_birds': [{'bird_id': 'B004', 'mutation': 'Rainbow', 'gender': 'Female', 'age': '7 Months', 'price': 4500, 'status': 'Available'}, {'bird_id': 'B005', 'mutation': 'Spangle', 'gender': 'Male', 'age': '5 Months', 'price': 3000, 'status': 'Available'}]}

BUDGIBIRD AI
Hello! We currently have a 7-month-old female Rainbow budgie (B004) for 4500 and a 5-month-old male Spangle budgie (B005) for 3000 available.


Conversation Length : 6
Token Usage : 1590 input | 253 output | Estimated Cost = $0.000777


In [58]:
run_agent(
    customer_id="C001",
    message="Where is my order?"
)

CUSTOMER : C001
QUESTION : Where is my order?

ROUTER
{'route': 'customer_support', 'category': 'order', 'urgency': 'medium', 'needs_human': False, 'summary': 'The user is inquiring about the status of their order.'}

SELECTED AGENT
Customer Support Agent

TOOL RESULT
{'agent': 'Customer Support Agent', 'customer': {'name': 'Minhaz', 'email': 'minhaz@gmail.com', 'phone': '017XXXXXXXX', 'address': 'Chattogram', 'birds': ['B001', 'B002'], 'orders': ['O001', 'O002']}, 'orders': [{'order_id': 'O001', 'customer_id': 'C001', 'bird_id': 'B001', 'status': 'Delivered'}, {'order_id': 'O002', 'customer_id': 'C001', 'bird_id': 'B002', 'status': 'Preparing'}]}

BUDGIBIRD AI
Hi Minhaz! Your order O001 has been delivered, and your order O002 is currently being prepared.


Conversation Length : 8
Token Usage : 2217 input | 323 output | Estimated Cost = $0.001039


In [59]:
run_agent(
    customer_id="C005",
    message="Where is my order?"
)

CUSTOMER : C005
QUESTION : Where is my order?

ROUTER
{'route': 'customer_support', 'category': 'order', 'urgency': 'medium', 'needs_human': False, 'summary': 'The user is requesting the status of their order.'}

SELECTED AGENT
Customer Support Agent

TOOL RESULT
{'agent': 'Customer Support Agent', 'status': 'Customer Not Found', 'message': 'Invalid Customer ID.'}

BUDGIBIRD AI
I'm sorry, I couldn't locate your order with that ID. Could you please double-check your information or contact our support team for further help?


Conversation Length : 10
Token Usage : 2710 input | 411 output | Estimated Cost = $0.001294


In [60]:
run_agent(
    customer_id="C001",
    message="I want a refund."
)

CUSTOMER : C001
QUESTION : I want a refund.

ROUTER
{'route': 'refund', 'category': 'refund', 'urgency': 'medium', 'needs_human': False, 'summary': 'The user is requesting a refund.'}

SELECTED AGENT
Refund Agent

TOOL RESULT
{'agent': 'Refund Agent', 'refunds': [{'refund_id': 'R001', 'customer_id': 'C001', 'order_id': 'O001', 'status': 'Pending'}], 'policy': 'Refund is available within 3 days with a valid avian veterinarian certificate.'}

BUDGIBIRD AI
Hello! Your refund request (R001) is currently pending; please provide a valid avian veterinarian certificate to complete the process within 3 days.


Conversation Length : 12
Token Usage : 3248 input | 480 output | Estimated Cost = $0.001532


In [66]:
run_agent(
    customer_id="C002",
    message="why i get twice bill?"
)

CUSTOMER : C002
QUESTION : why i get twice bill?

ROUTER
{'route': 'customer_support', 'category': 'order', 'urgency': 'medium', 'needs_human': True, 'summary': 'The user is inquiring about a duplicate billing issue on their order.'}

SELECTED AGENT
Customer Support Agent

TOOL RESULT
{'agent': 'Customer Support Agent', 'customer': {'name': 'Rahim', 'email': 'rahim@gmail.com', 'phone': '018XXXXXXXX', 'address': 'Dhaka', 'birds': ['B003'], 'orders': ['O003']}, 'orders': [{'order_id': 'O003', 'customer_id': 'C002', 'bird_id': 'B003', 'status': 'Delivered'}]}

BUDGIBIRD AI
Hi Rahim, I see only one order (O003) for your bird. Please contact us with your payment receipt so I can investigate this for you.


Conversation Length : 24
Token Usage : 6400 input | 931 output | Estimated Cost = $0.002997


In [62]:
run_agent(
    customer_id="C002",
    message="what i have purchesd?"
)

CUSTOMER : C002
QUESTION : what i have purchesd?

ROUTER
{'route': 'customer_support', 'category': 'order', 'urgency': 'low', 'needs_human': False, 'summary': 'The user is inquiring about their purchase history or current order details.'}

SELECTED AGENT
Customer Support Agent

TOOL RESULT
{'agent': 'Customer Support Agent', 'customer': {'name': 'Rahim', 'email': 'rahim@gmail.com', 'phone': '018XXXXXXXX', 'address': 'Dhaka', 'birds': ['B003'], 'orders': ['O003']}, 'orders': [{'order_id': 'O003', 'customer_id': 'C002', 'bird_id': 'B003', 'status': 'Delivered'}]}

BUDGIBIRD AI
Hi Rahim! You purchased bird B003, and your order O003 has been successfully delivered.


Conversation Length : 16
Token Usage : 4324 input | 631 output | Estimated Cost = $0.002027


In [73]:
run_agent(
    message="Analyze this budgie. and i want to contact vet",
    image_path= "Injured birds.jpg"
)

CUSTOMER : Guest User
QUESTION : Analyze this budgie. and i want to contact vet

ROUTER
{'route': 'emergency', 'category': 'emergency', 'urgency': 'high', 'needs_human': True, 'summary': 'User requested analysis of a bird image and expressed a need to contact a veterinarian.'}

SELECTED AGENT
Emergency Agent

TOOL RESULT
{'agent': 'Emergency Agent', 'emergency': {'status': 'Emergency', 'action': 'Visit an avian veterinarian immediately.'}, 'possible_cause': 'No disease information available.'}

BUDGIBIRD AI
Your budgie appears to have visible bleeding on its wing; please visit an avian veterinarian immediately as this is an emergency.


Conversation Length : 33
Token Usage : 12991 input | 1323 output | Estimated Cost = $0.005232
